# 🎧 MixIT (Mixture Invariant Training) - Google Colab 클라우드 학습 & 추론 노트북

이 노트북은 **MixIT (`neurips2020_mixit`)** 비지도 음원 분리 모델을 Google Colab에서 학습, 모니터링, 평가 및 추론할 수 있도록 구성된 통합 워크플로우입니다.

### 📌 핵심 아키텍처 (스토리지 분리 전략)
- **데이터셋 (`FUSS`)**: I/O 병목 방지 및 Google Drive 용량 절약을 위해 **Colab 로컬 디스크 (`/content/fuss_data`)**에 고속 다운로드 (`aria2c` 병렬 다운로드 후 압축파일 자동 삭제)
- **체크포인트 & TensorBoard 로그**: Colab 세션이 끊겨도 안전하게 보존되고 **자동으로 이어서 학습(Resume)**할 수 있도록 **Google Drive (`/content/drive/MyDrive/sound_separation/mixit_fuss`)**에 실시간 저장

## 1. GPU 런타임 확인 및 Google Drive 마운트

In [ ]:
# GPU 할당 확인 (런타임 -> 런타임 유형 변경 -> T4 / L4 / A100 GPU 선택 필수)
!nvidia-smi

In [ ]:
import os
from google.colab import drive

# Google Drive 마운트
drive.mount('/content/drive')

# 체크포인트 및 로그를 영구 보관할 Google Drive 경로 설정
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sound_separation'
EXP_NAME = 'mixit_fuss_exp1'  # 이어서 학습할 때는 동일한 실험 이름 유지
MODEL_DIR = os.path.join(DRIVE_PROJECT_DIR, 'checkpoints', EXP_NAME)

os.makedirs(MODEL_DIR, exist_ok=True)
print(f'✅ 체크포인트 및 TensorBoard 로그 저장 경로: {MODEL_DIR}')

## 2. 프로젝트 코드 준비 및 Colab 호환 환경 설정
- 프로젝트를 **GitHub 저장소에서 클론**하거나, **Google Drive에 업로드된 `sound-separation` 폴더**를 `/content/sound-separation`으로 복사합니다.

In [ ]:
import os
import shutil

REPO_DIR = '/content/sound-separation'

# 방법 A: 본인의 GitHub 저장소 URL 사용 시 아래 주소 변경
GIT_REPO_URL = 'https://github.com/yhcho1443/sound-separation.git'

# 방법 B: Google Drive에 sound-separation 폴더를 업로드해 둔 경우 경로 지정
DRIVE_REPO_SOURCE = '/content/drive/MyDrive/sound-separation'

if not os.path.exists(REPO_DIR):
    if GIT_REPO_URL:
        print(f'Git 저장소에서 클론 중: {GIT_REPO_URL}')
        !git clone {GIT_REPO_URL} {REPO_DIR}
    elif os.path.exists(DRIVE_REPO_SOURCE):
        print(f'Google Drive({DRIVE_REPO_SOURCE})에서 로컬(/content)로 코드 복사 중...')
        shutil.copytree(DRIVE_REPO_SOURCE, REPO_DIR, dirs_exist_ok=True)
    else:
        print('⚠️ 기본 google-research/sound-separation 저장소를 클론합니다.')
        print('   (주의: 로컬에서 패치한 colab/ 폴더와 호환성 코드가 포함된 저장소나 Drive 폴더를 사용하는 것을 권장합니다.)')
        !git clone https://github.com/google-research/sound-separation.git {REPO_DIR}

%cd {REPO_DIR}
!ls -la

In [ ]:
# Colab 필수 패키지 및 TF v1 / Legacy Keras 호환 환경 설치
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

!bash colab/setup_colab.sh

## 3. 데이터셋 준비 (Colab 로컬 `/content/fuss_data`)
- **옵션 A (빠른 동작 확인용 더미 데이터 - 5초 소요)**: 전체 6.4GB 데이터를 받기 전에 학습/평가 파이프라인이 정상 작동하는지 즉시 테스트합니다.
- **옵션 B (실제 FUSS 데이터셋 다운로드 - 약 6.4GB)**: Zenodo에서 `FUSS_ssdata.tar.gz`를 `/content/fuss_data`에 멀티스레드로 다운로드 및 압축 해제한 뒤 `.tar.gz` 원본을 자동 삭제하여 Colab 디스크 공간을 확보합니다.

In [ ]:
# [옵션 Selection] USE_DUMMY_DATA = True 이면 5초 만에 생성되는 테스트용 미니 데이터셋 사용,
# 실제 본 학습을 진행할 때는 False 로 변경하세요!
USE_DUMMY_DATA = False

if USE_DUMMY_DATA:
    DATA_DIR = '/content/fuss_dummy/ssdata'
    !python3 colab/create_dummy_fuss.py --data_dir {DATA_DIR}
else:
    FUSS_ROOT_DIR = '/content/fuss_data'
    DATA_DIR = f'{FUSS_ROOT_DIR}/fuss_dev/ssdata'
    !bash colab/download_fuss_local.sh --root-dir {FUSS_ROOT_DIR}

print(f'✅ 학습에 사용할 DATA_DIR: {DATA_DIR}')
!ls -la {DATA_DIR}

## 4. TensorBoard 실시간 모니터링 실행
- 학습 시작 전 아래 셀을 실행해 두면 손실(`loss_mixit`), 분리 성능(`SNR`), 스펙트로그램 및 분리된 오디오 샘플을 실시간으로 확인할 수 있습니다.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{MODEL_DIR}"

## 5. MixIT 모델 학습 실행 (자동 체크포인트 저장 & 이어하기)
- `MODEL_DIR`에 이미 체크포인트(`model.ckpt-*`)가 존재하면 **자동으로 마지막 스텝부터 이어서 학습(Resume)**합니다.
- **배치 사이즈 팁**:
  - `BATCH_SIZE = 3` (논문 기본값, MoM 생성을 위해 실제 입력 배치는 `2 * 3 = 6`개 오디오 사용)
  - Colab 무료 T4 GPU(15GB VRAM)에서 OOM 발생 시 `BATCH_SIZE = 2` 또는 `1`로 낮추세요.

In [ ]:
# 하이퍼파라미터 설정
BATCH_SIZE = 2            # T4 GPU 권장: 2 (실제 배치 4), A100/L4 권장: 3 (실제 배치 6)
TRAIN_STEPS = 20000000    # 스모크 테스트 시 200 등으로 설정
EVAL_EXAMPLES = 200       # 검증 샘플 수 (기본 800, 빠른 검증 시 200)
SAVE_CKPT_SECS = 600      # 10분(600초)마다 Google Drive에 체크포인트 자동 저장
SAVE_SUMMARY_STEPS = 500  # 500 스텝마다 TensorBoard 요약 기록
PARALLEL_READERS = 16     # Colab RAM(12.7GB) OOM 방지를 위한 병렬 리더 수
LEARNING_RATE = 1e-4

!TF_USE_LEGACY_KERAS=1 python3 models/neurips2020_mixit/train_model_on_fuss.py \
    --data_dir "{DATA_DIR}" \
    --model_dir "{MODEL_DIR}" \
    --batch_size {BATCH_SIZE} \
    --train_steps {TRAIN_STEPS} \
    --eval_examples {EVAL_EXAMPLES} \
    --save_checkpoints_secs {SAVE_CKPT_SECS} \
    --save_summary_steps {SAVE_SUMMARY_STEPS} \
    --parallel_readers {PARALLEL_READERS} \
    --lr {LEARNING_RATE}

## 6. 오디오 음원 분리 추론 (Inference) 및 바로 듣기
- 직접 학습한 체크포인트(`MODEL_DIR`) 또는 Google Research의 사전 학습된 YFCC100m MixIT 체크포인트를 이용해 혼합 오디오(`.wav`)를 8개 채널로 분리하고 노트북에서 바로 들어봅니다.

In [ ]:
# (선택) Google Research 사전 학습된 YFCC100m MixIT 체크포인트 다운로드
USE_PRETRAINED_YFCC100M = False

if USE_PRETRAINED_YFCC100M:
    PRETRAINED_DIR = '/content/yfcc100m_mixit_model_checkpoint'
    if not os.path.exists(PRETRAINED_DIR):
        !gsutil -m cp -r gs://gresearch/sound_separation/yfcc100m_mixit_model_checkpoint /content/
    INFER_MODEL_DIR = PRETRAINED_DIR
else:
    INFER_MODEL_DIR = MODEL_DIR

print(f'추론에 사용할 모델 경로: {INFER_MODEL_DIR}')
!ls -la "{INFER_MODEL_DIR}"

In [ ]:
import glob
import IPython.display as ipd

# 테스트할 샘플 wav 파일 선택 (DATA_DIR의 eval 셋 중 첫 번째 파일 또는 직접 업로드한 파일)
eval_wavs = sorted(glob.glob(f'{DATA_DIR}/eval/*.wav'))
INPUT_WAV = eval_wavs[0] if eval_wavs else ''
OUTPUT_DIR = '/content/separated_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_WAV = os.path.join(OUTPUT_DIR, 'separated.wav')

assert INPUT_WAV and os.path.exists(INPUT_WAV), '분리할 입력 WAV 파일 경로를 확인하세요!'
print(f'입력 혼합 오디오: {INPUT_WAV}')
ipd.display(ipd.Audio(INPUT_WAV))

!TF_USE_LEGACY_KERAS=1 PYTHONPATH=models:models/tools python3 models/tools/process_wav.py \
    --model_dir "{INFER_MODEL_DIR}" \
    --input "{INPUT_WAV}" \
    --output "{OUTPUT_WAV}" \
    --num_sources 8 \
    --write_outputs_separately True

print('\n=== 🔊 분리된 소스 오디오 결과 ===')
for src_wav in sorted(glob.glob(os.path.join(OUTPUT_DIR, 'separated_source*.wav'))):
    print(os.path.basename(src_wav))
    ipd.display(ipd.Audio(src_wav))

## 7. FUSS 테스트셋 정량 평가 (SI-SNR / SI-SNRi 측정)
- 학습된 체크포인트의 분리 성능(SI-SNR, SI-SNRi, Under/Equal/Over-separation)을 측정하고 결과 CSV 및 요약 리포트를 Google Drive에 저장합니다.

In [ ]:
import tensorflow.compat.v1 as tf

LATEST_CKPT = tf.train.latest_checkpoint(INFER_MODEL_DIR)
METAGRAPH_PATH = os.path.join(INFER_MODEL_DIR, 'inference.meta')
EVAL_LIST_PATH = os.path.join(DATA_DIR, 'eval_example_list.txt')
EVAL_OUT_DIR = os.path.join(INFER_MODEL_DIR, 'eval_results')
os.makedirs(EVAL_OUT_DIR, exist_ok=True)

print(f'평가 대상 체크포인트: {LATEST_CKPT}')

!TF_USE_LEGACY_KERAS=1 python3 models/dcase2020_fuss_baseline/evaluate.py \
    --checkpoint_path "{LATEST_CKPT}" \
    --metagraph_path "{METAGRAPH_PATH}" \
    --data_list_path "{EVAL_LIST_PATH}" \
    --output_path "{EVAL_OUT_DIR}/scores.csv"